In [ ]:
# =================================================================================================
# CLEAN GAZE-ALIGNED ECG EXPERIMENT
# =================================================================================================
#
# PURPOSE
# -------
# A fully controlled corrective experiment in which:
#
#   BASE              = original 12 leads.h5
#   CONSULTANT        = gaze supervision from 9 consultants
#   ALL_CLINICIANS    = gaze supervision from 9 consultants + 10 fellows + 1 resident
#
# BOTH gaze models use:
#   - exactly the same ECG image preprocessing
#   - 12 separately cropped leads
#   - correct standard lead order:
#         I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6
#   - equal weighting of readers
#   - gaze-only MSE objective
#   - only conv2d_10 unfrozen
#   - frozen BatchNorm behaviour
#   - identical optimiser / epochs / augmentation / seed
#
# VALIDATION:
#   1. 5-fold ECG-level held-out gaze-alignment validation on the 10 gaze ECGs
#   2. Final training on all 10 ECGs
#   3. Internal PTB-XL representation probing
#        - MI vs NORM
#        - AFIB vs SR
#        - 5-fold patient-grouped OOF evaluation
#   4. External LUDB validation
#        - AF
#        - ischaemia-related annotation
#        - conduction abnormality
#        - LVH
#        - paced rhythm
#        - 5-fold OOF evaluation
#   5. Paired bootstrap effect-size CIs
#   6. Exact McNemar tests
#   7. Holm adjustment of McNemar p-values
#
# IMPORTANT:
# Inspect the QC_*.png files generated from the 10 source ECG images.
# If the lead crops are visibly wrong, adjust GRID_BOUNDS_FRAC/CELL_PADDING
# before interpreting any results.
#
# =================================================================================================


# =================================================================================================
# 0. REPRODUCIBILITY + IMPORTS
# =================================================================================================

import os
os.environ["TF_DETERMINISTIC_OPS"] = "1"

import gc
import ast
import glob
import json
import math
import random
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf
import wfdb

from PIL import Image

import matplotlib.pyplot as plt

from scipy.signal import savgol_filter
from scipy.stats import wilcoxon, binomtest

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    confusion_matrix,
)
from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
)

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAVE_STRATIFIED_GROUP_KFOLD = True
except ImportError:
    HAVE_STRATIFIED_GROUP_KFOLD = False

from tensorflow import keras
from tensorflow.keras import Model


warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)


# =================================================================================================
# 1. PATHS — EDIT ONLY IF YOUR FOLDERS DIFFER
# =================================================================================================

BASE_MODEL_PATH = "12 leads.h5"

PTB_ROOT = "ptb-xl"

LUDB_ROOT = "Test ECG"
LUDB_CSV = os.path.join(LUDB_ROOT, "ludb.csv")
LUDB_DATA = os.path.join(LUDB_ROOT, "data")

EYE_ROOT = (
    "eye_tracking/"
    "eye-tracking-dataset-for-the-12-lead-electrocardiogram-"
    "interpretation-of-medical-practitioners-and-students-1.0.0"
)

GRID_PATH = os.path.join(
    EYE_ROOT,
    "Datasets",
    "Grid_Anonymized.csv"
)

OUT_DIR = "clean_gaze_experiment"
os.makedirs(OUT_DIR, exist_ok=True)


# =================================================================================================
# 2. PRE-SPECIFIED EXPERIMENT SETTINGS
# =================================================================================================

STANDARD_LEADS = [
    "I", "II", "III",
    "aVR", "aVL", "aVF",
    "V1", "V2", "V3",
    "V4", "V5", "V6",
]

# Physical display arrangement of the ECG images:
DISPLAY_LEADS = [
    ["I",   "aVR", "V1", "V4"],
    ["II",  "aVL", "V2", "V5"],
    ["III", "aVF", "V3", "V6"],
]

ATTENTION_LAYER = "add_3"

# Only this convolution is allowed to update.
TRAINABLE_LAYER = "conv2d_10"

LEARNING_RATE = 1e-4
EPOCHS = 25

# Original consultant implementation used 40 noise copies.
# We retain this augmentation scale but apply it identically to both clean models.
AUGMENTATIONS_PER_ECG = 40
NOISE_SD = 0.03
TRAIN_BATCH_SIZE = 16

N_GAZE_FOLDS = 5
N_EVAL_FOLDS = 5

# Paired bootstrap repetitions
BOOT_N = 5000

FEATURE_LAYER = "dense_1"

# Prediction batch size
PRED_BATCH_SIZE = 64


# -------------------------------------------------------------------------------------------------
# ECG IMAGE GRID PARAMETERS
#
# Fractions are: left, top, right, bottom.
#
# Default assumes the 3x4 ECG grid occupies almost the entire image.
# Inspect the saved QC figures.
# -------------------------------------------------------------------------------------------------

GRID_BOUNDS_FRAC = (
    0.00,   # left
    0.00,   # top
    1.00,   # right
    1.00,   # bottom
)

# Small padding inside each lead cell to reduce labels/borders.
CELL_PAD_X = 0.03
CELL_PAD_Y = 0.05

CELL_RESIZE_HEIGHT = 128
CELL_RESIZE_WIDTH = 1000


# =================================================================================================
# 3. GAZE ECG DEFINITIONS
# =================================================================================================

ECG_KEYS = [
    "NSR",
    "AFib",
    "Hyperkalemia",
    "AFlutter",
    "VTach",
    "WPW",
    "VentPaced",
    "LBBB",
    "STEMI",
    "CompleteHeartBlock",
]

PARENT_TO_KEY = {
    "normal sinus rhythm": "NSR",
    "atrial fibrillation": "AFib",
    "hyperkalemia": "Hyperkalemia",
    "atrial flutter": "AFlutter",
    "ventricular tachycardia": "VTach",
    "wolf parkinson white syndrome": "WPW",
    "ventricular paced rhythm": "VentPaced",
    "left bundle branch block": "LBBB",
    "st elevation mi": "STEMI",
    "complete heart block": "CompleteHeartBlock",
}

IMAGE_FILES = {
    "NSR":
        "Normal_Sinus_Rhythm.jpg",

    "AFib":
        "Atrial_Fibrillation.png",

    "Hyperkalemia":
        "Hyperkalemia.jpg",

    "AFlutter":
        "Atrial_Flutter.jpg",

    "VTach":
        "Ventricular_Tachycardia.jpg",

    "WPW":
        "Wolf_Parkinson_White_Syndrome.jpg",

    "VentPaced":
        "Ventricular_Paced_Rhythm.jpg",

    "LBBB":
        "Left_Bundle_Branch_Block.jpg",

    "STEMI":
        "STEMI.png",

    "CompleteHeartBlock":
        "Complete_Heart_Block.jpg",
}

LABEL_TO_STANDARD = {
    "1": "I",
    "2": "II",
    "3": "III",

    "I": "I",
    "II": "II",
    "III": "III",

    "aVR": "aVR",
    "aVL": "aVL",
    "aVF": "aVF",

    "V1": "V1",
    "V2": "V2",
    "V3": "V3",
    "V4": "V4",
    "V5": "V5",
    "V6": "V6",
}


# =================================================================================================
# 4. HELPER FUNCTIONS
# =================================================================================================

def find_file(filename):
    """Find source ECG image recursively."""
    search_roots = [
        EYE_ROOT,
        ".",
    ]

    matches = []

    for root in search_roots:
        matches.extend(
            glob.glob(
                os.path.join(
                    root,
                    "**",
                    filename
                ),
                recursive=True
            )
        )

    matches = list(dict.fromkeys(matches))

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find {filename}"
        )

    if len(matches) > 1:
        print(
            f"WARNING: multiple matches for {filename}; "
            f"using {matches[0]}"
        )

    return matches[0]


def zscore_1d(x):
    x = np.asarray(x, dtype=np.float32)
    return (
        x - np.mean(x)
    ) / (
        np.std(x) + 1e-8
    )


def resize_gray(arr, width, height):
    arr8 = np.clip(
        arr * 255.0,
        0,
        255
    ).astype(np.uint8)

    pil = Image.fromarray(arr8)

    try:
        resample = Image.Resampling.BILINEAR
    except AttributeError:
        resample = Image.BILINEAR

    pil = pil.resize(
        (width, height),
        resample=resample
    )

    return (
        np.asarray(pil)
        .astype(np.float32)
        / 255.0
    )


def extract_trace_from_cell(cell):
    """
    Convert one lead image crop into an approximate 1-D waveform.

    Method:
      1. resize to fixed resolution
      2. invert image -> darkness
      3. suppress row/column background structure
      4. estimate vertical centre-of-mass of residual dark trace
      5. interpolate poorly supported columns
      6. smooth
      7. z-normalise

    This remains an IMAGE-DERIVED PSEUDO-SIGNAL, not true digitisation.
    """

    cell = resize_gray(
        cell,
        CELL_RESIZE_WIDTH,
        CELL_RESIZE_HEIGHT
    )

    darkness = 1.0 - cell

    # Remove column-wise background (helps suppress vertical grid lines)
    darkness = (
        darkness
        - np.median(
            darkness,
            axis=0,
            keepdims=True
        )
    )

    # Remove row-wise background (helps suppress horizontal grid lines)
    darkness = (
        darkness
        - np.median(
            darkness,
            axis=1,
            keepdims=True
        )
    )

    darkness = np.clip(
        darkness,
        0.0,
        None
    )

    # Emphasise strongest residual dark structures.
    threshold = np.quantile(
        darkness,
        0.75,
        axis=0,
        keepdims=True
    )

    weights = np.where(
        darkness >= threshold,
        darkness,
        0.0
    )

    weights = weights ** 2

    y_coords = np.arange(
        CELL_RESIZE_HEIGHT,
        dtype=np.float32
    )[:, None]

    denominator = np.sum(
        weights,
        axis=0
    )

    numerator = np.sum(
        weights * y_coords,
        axis=0
    )

    centroid = np.full(
        CELL_RESIZE_WIDTH,
        np.nan,
        dtype=np.float32
    )

    valid = denominator > 1e-8

    centroid[valid] = (
        numerator[valid]
        / denominator[valid]
    )

    # Interpolate unsupported columns
    valid_idx = np.where(
        np.isfinite(centroid)
    )[0]

    if len(valid_idx) < 10:
        # Conservative fallback: vertical intensity projection
        trace = 1.0 - np.mean(
            cell,
            axis=0
        )

        return zscore_1d(trace)

    missing_idx = np.where(
        ~np.isfinite(centroid)
    )[0]

    if len(missing_idx) > 0:
        centroid[missing_idx] = np.interp(
            missing_idx,
            valid_idx,
            centroid[valid_idx]
        )

    # Image y increases downwards, so invert sign.
    trace = -centroid

    # Mild smoothing
    try:
        trace = savgol_filter(
            trace,
            window_length=11,
            polyorder=2,
            mode="interp"
        )
    except Exception:
        pass

    return zscore_1d(trace)


def image_to_standard_12lead(path):
    """
    Divide image into 3 x 4 displayed lead cells,
    reconstruct each lead separately,
    then reorder into STANDARD waveform order.
    """

    img = (
        np.asarray(
            Image.open(path)
            .convert("L")
        )
        .astype(np.float32)
        / 255.0
    )

    H, W = img.shape

    left_f, top_f, right_f, bottom_f = (
        GRID_BOUNDS_FRAC
    )

    x0 = int(W * left_f)
    x1 = int(W * right_f)

    y0 = int(H * top_f)
    y1 = int(H * bottom_f)

    grid_img = img[
        y0:y1,
        x0:x1
    ]

    GH, GW = grid_img.shape

    traces = {}

    for row in range(3):
        for col in range(4):

            lead = DISPLAY_LEADS[row][col]

            cy0 = int(
                row * GH / 3
            )

            cy1 = int(
                (row + 1) * GH / 3
            )

            cx0 = int(
                col * GW / 4
            )

            cx1 = int(
                (col + 1) * GW / 4
            )

            cell = grid_img[
                cy0:cy1,
                cx0:cx1
            ]

            ch, cw = cell.shape

            px = int(
                CELL_PAD_X * cw
            )

            py = int(
                CELL_PAD_Y * ch
            )

            if (
                2 * px < cw
                and
                2 * py < ch
            ):
                cell = cell[
                    py:ch-py,
                    px:cw-px
                ]

            traces[lead] = (
                extract_trace_from_cell(
                    cell
                )
            )

    X = np.stack(
        [
            traces[lead]
            for lead in STANDARD_LEADS
        ],
        axis=0
    )

    return (
        X[..., None]
        .astype(np.float32)
    )


def feature_to_lead_attention(feature_tensor):
    """
    Convert add_3 representation to 12-element distribution.

    Expected possible shapes:
       [B,12,T,C]
       [B,12,T]
       [B,12]
    """

    rank = feature_tensor.shape.rank

    x = tf.abs(
        feature_tensor
    )

    if rank == 4:
        x = tf.reduce_mean(
            x,
            axis=[2, 3]
        )

    elif rank == 3:
        x = tf.reduce_mean(
            x,
            axis=2
        )

    elif rank == 2:
        pass

    else:
        raise ValueError(
            f"Unexpected attention tensor rank: {rank}"
        )

    x = x + 1e-8

    x = (
        x
        / tf.reduce_sum(
            x,
            axis=1,
            keepdims=True
        )
    )

    return x


def cosine_similarity_np(a, b):

    a = np.asarray(a, float)
    b = np.asarray(b, float)

    return float(
        np.dot(a, b)
        /
        (
            np.linalg.norm(a)
            *
            np.linalg.norm(b)
            + 1e-12
        )
    )


def mse_np(a, b):

    return float(
        np.mean(
            (
                np.asarray(a)
                -
                np.asarray(b)
            ) ** 2
        )
    )


def js_divergence_np(p, q):

    p = np.asarray(
        p,
        dtype=float
    )

    q = np.asarray(
        q,
        dtype=float
    )

    p = (
        p + 1e-12
    )

    q = (
        q + 1e-12
    )

    p = p / p.sum()
    q = q / q.sum()

    m = 0.5 * (
        p + q
    )

    kl_pm = np.sum(
        p * np.log(
            p / m
        )
    )

    kl_qm = np.sum(
        q * np.log(
            q / m
        )
    )

    return float(
        0.5 * (
            kl_pm + kl_qm
        )
    )


# =================================================================================================
# 5. LOAD AND AUDIT GAZE DATA
# =================================================================================================

print(
    "\n" + "=" * 100
)

print(
    "GAZE DATA"
)

print(
    "=" * 100
)

grid = pd.read_csv(
    GRID_PATH
)

grid["ECG_Key"] = (
    grid["ParentStimulus"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(PARENT_TO_KEY)
)

grid["LeadToken"] = (
    grid["Label"]
    .astype(str)
    .str.split()
    .str[0]
)

grid["LeadStd"] = (
    grid["LeadToken"]
    .map(LABEL_TO_STANDARD)
)

grid = grid[
    (
        grid["Type"]
        ==
        "Static AOI"
    )
    &
    grid["ECG_Key"].notna()
    &
    grid["LeadStd"].notna()
].copy()


print(
    "Unique readers:"
)

print(
    grid.groupby(
        "Group"
    )["Respondent_Name"]
    .nunique()
)


# =================================================================================================
# 6. CREATE EQUAL-READER-WEIGHTED GAZE TARGETS
# =================================================================================================

CONSULTANT_GROUPS = [
    "Consultant"
]

ALL_CLINICIAN_GROUPS = [
    "Consultant",
    "Fellow",
    "resident",
]


def build_equal_reader_target(groups):

    subset = grid[
        grid["Group"].isin(
            groups
        )
    ].copy()

    targets = {}
    contributor_counts = {}

    for ecg_key in ECG_KEYS:

        ecg = subset[
            subset["ECG_Key"]
            ==
            ecg_key
        ]

        reader_vectors = []

        readers_used = []

        for respondent, rdf in (
            ecg.groupby(
                "Respondent_Name"
            )
        ):

            sums = (
                rdf.groupby(
                    "LeadStd"
                )["Time_spent_G"]
                .sum()
            )

            vec = np.array(
                [
                    float(
                        sums.get(
                            lead,
                            0.0
                        )
                    )
                    for lead
                    in STANDARD_LEADS
                ],
                dtype=np.float32
            )

            if (
                np.sum(vec)
                >
                0
            ):
                vec = (
                    vec
                    /
                    np.sum(vec)
                )

                reader_vectors.append(
                    vec
                )

                readers_used.append(
                    respondent
                )

        if (
            len(
                reader_vectors
            )
            ==
            0
        ):
            raise RuntimeError(
                f"No gaze data for {ecg_key}"
            )

        target = np.mean(
            np.stack(
                reader_vectors
            ),
            axis=0
        )

        target = (
            target
            /
            target.sum()
        )

        targets[
            ecg_key
        ] = (
            target
            .astype(
                np.float32
            )
        )

        contributor_counts[
            ecg_key
        ] = len(
            readers_used
        )

    return (
        targets,
        contributor_counts
    )


consultant_targets, consultant_counts = (
    build_equal_reader_target(
        CONSULTANT_GROUPS
    )
)

all_clinician_targets, all_clinician_counts = (
    build_equal_reader_target(
        ALL_CLINICIAN_GROUPS
    )
)


print(
    "\nConsultant target readers:"
)

print(
    consultant_counts
)

print(
    "\nAll-clinician target readers:"
)

print(
    all_clinician_counts
)


assert all(
    n == 9
    for n
    in consultant_counts.values()
), (
    "Expected 9 consultants for every ECG"
)

assert all(
    n == 20
    for n
    in all_clinician_counts.values()
), (
    "Expected 20 clinical readers for every ECG"
)


# Save gaze targets
gaze_target_rows = []

for target_name, targets in [
    (
        "CONSULTANT",
        consultant_targets
    ),
    (
        "ALL_CLINICIANS",
        all_clinician_targets
    ),
]:

    for ecg in ECG_KEYS:

        for lead, value in zip(
            STANDARD_LEADS,
            targets[ecg]
        ):

            gaze_target_rows.append(
                {
                    "target":
                        target_name,

                    "ECG":
                        ecg,

                    "lead":
                        lead,

                    "weight":
                        value,
                }
            )

pd.DataFrame(
    gaze_target_rows
).to_csv(
    os.path.join(
        OUT_DIR,
        "gaze_targets_equal_reader_weighted.csv"
    ),
    index=False
)


# =================================================================================================
# 7. LOAD + RECONSTRUCT THE TEN GAZE ECG IMAGES
# =================================================================================================

print(
    "\n" + "=" * 100
)

print(
    "IMAGE-DERIVED PSEUDO-SIGNALS"
)

print(
    "=" * 100
)

image_paths = {}

X_gaze_list = []

for ecg_key in ECG_KEYS:

    path = find_file(
        IMAGE_FILES[
            ecg_key
        ]
    )

    image_paths[
        ecg_key
    ] = path

    print(
        ecg_key,
        "->",
        path
    )

    X_ecg = (
        image_to_standard_12lead(
            path
        )
    )

    X_gaze_list.append(
        X_ecg
    )

X_gaze = np.stack(
    X_gaze_list
)

print(
    "X_gaze:",
    X_gaze.shape
)

assert (
    X_gaze.shape
    ==
    (
        10,
        12,
        1000,
        1
    )
)


# =================================================================================================
# 8. SAVE QC FIGURES — INSPECT THESE
# =================================================================================================

for i, ecg_key in enumerate(
    ECG_KEYS
):

    fig, axes = plt.subplots(
        3,
        4,
        figsize=(
            14,
            8
        ),
        sharex=True
    )

    axes = axes.ravel()

    for j, lead in enumerate(
        STANDARD_LEADS
    ):

        axes[j].plot(
            X_gaze[
                i,
                j,
                :,
                0
            ]
        )

        axes[j].set_title(
            lead
        )

        axes[j].set_xticks([])
        axes[j].set_yticks([])

    fig.suptitle(
        f"{ecg_key}: reconstructed image-derived pseudo-signals"
    )

    fig.tight_layout()

    fig.savefig(
        os.path.join(
            OUT_DIR,
            f"QC_{ecg_key}.png"
        ),
        dpi=180
    )

    plt.close(
        fig
    )

print(
    "\nSaved ten QC_*.png figures."
)

print(
    "*** INSPECT THESE BEFORE ACCEPTING THE FINAL EXPERIMENT. ***"
)


# =================================================================================================
# 9. MODEL BUILDING + CLEAN GAZE TRAINING
# =================================================================================================

def fresh_base_model():

    model = (
        keras.models.load_model(
            BASE_MODEL_PATH,
            compile=False
        )
    )

    return model


def prepare_clean_gaze_model():

    model = fresh_base_model()

    for layer in model.layers:
        layer.trainable = False

    model.get_layer(
        TRAINABLE_LAYER
    ).trainable = True

    attention_model = Model(
        inputs=model.input,
        outputs=model.get_layer(
            ATTENTION_LAYER
        ).output
    )

    trainable_names = [
        v.name
        for v
        in attention_model.trainable_variables
    ]

    print(
        "Trainable variables:",
        trainable_names
    )

    if (
        len(
            attention_model.trainable_variables
        )
        ==
        0
    ):
        raise RuntimeError(
            "No trainable variables found"
        )

    return (
        model,
        attention_model
    )


def augment_training_data(
    X,
    Y,
    seed
):

    rng = (
        np.random.default_rng(
            seed
        )
    )

    Xa = []
    Ya = []

    for i in range(
        len(X)
    ):

        # Always include exact original
        Xa.append(
            X[i].copy()
        )

        Ya.append(
            Y[i].copy()
        )

        for _ in range(
            AUGMENTATIONS_PER_ECG - 1
        ):

            noise = rng.normal(
                loc=0.0,
                scale=NOISE_SD,
                size=X[i].shape
            ).astype(
                np.float32
            )

            Xa.append(
                X[i]
                +
                noise
            )

            Ya.append(
                Y[i].copy()
            )

    return (
        np.stack(
            Xa
        ).astype(
            np.float32
        ),
        np.stack(
            Ya
        ).astype(
            np.float32
        )
    )


def train_clean_gaze_model(
    X,
    Y,
    seed
):

    tf.keras.utils.set_random_seed(
        seed
    )

    model, attention_model = (
        prepare_clean_gaze_model()
    )

    Xa, Ya = (
        augment_training_data(
            X,
            Y,
            seed
        )
    )

    optimizer = (
        keras.optimizers.Adam(
            learning_rate=
            LEARNING_RATE
        )
    )

    rng = (
        np.random.default_rng(
            seed
        )
    )

    history = []

    n = len(
        Xa
    )

    for epoch in range(
        EPOCHS
    ):

        order = (
            rng.permutation(
                n
            )
        )

        epoch_losses = []

        for start in range(
            0,
            n,
            TRAIN_BATCH_SIZE
        ):

            idx = order[
                start:
                start
                +
                TRAIN_BATCH_SIZE
            ]

            xb = tf.convert_to_tensor(
                Xa[idx],
                dtype=tf.float32
            )

            yb = tf.convert_to_tensor(
                Ya[idx],
                dtype=tf.float32
            )

            with tf.GradientTape() as tape:

                feat = attention_model(
                    xb,
                    training=False
                )

                pred_attention = (
                    feature_to_lead_attention(
                        feat
                    )
                )

                # CLEAN GAZE-ONLY OBJECTIVE
                loss = tf.reduce_mean(
                    tf.reduce_mean(
                        tf.square(
                            pred_attention
                            -
                            yb
                        ),
                        axis=1
                    )
                )

            variables = (
                attention_model
                .trainable_variables
            )

            grads = tape.gradient(
                loss,
                variables
            )

            usable = [
                (
                    g,
                    v
                )
                for g, v
                in zip(
                    grads,
                    variables
                )
                if g is not None
            ]

            if (
                len(
                    usable
                )
                ==
                0
            ):
                raise RuntimeError(
                    "No gradients reached trainable layer."
                )

            optimizer.apply_gradients(
                usable
            )

            epoch_losses.append(
                float(
                    loss.numpy()
                )
            )

        history.append(
            float(
                np.mean(
                    epoch_losses
                )
            )
        )

    return (
        model,
        history
    )


def get_attention_distribution(
    model,
    X
):

    attention_model = Model(
        inputs=model.input,
        outputs=model.get_layer(
            ATTENTION_LAYER
        ).output
    )

    feat = attention_model(
        X,
        training=False
    )

    att = (
        feature_to_lead_attention(
            feat
        )
        .numpy()
    )

    return att


# =================================================================================================
# 10. BASE ATTENTION ON THE TEN GAZE ECGs
# =================================================================================================

base_for_gaze = fresh_base_model()

base_gaze_attention = (
    get_attention_distribution(
        base_for_gaze,
        X_gaze
    )
)

del base_for_gaze
gc.collect()


# =================================================================================================
# 11. 5-FOLD ECG-LEVEL HELD-OUT GAZE-ALIGNMENT VALIDATION
# =================================================================================================

print(
    "\n" + "=" * 100
)

print(
    "5-FOLD HELD-OUT GAZE ALIGNMENT VALIDATION"
)

print(
    "=" * 100
)

TARGETS = {
    "CONSULTANT":
        np.stack(
            [
                consultant_targets[
                    k
                ]
                for k
                in ECG_KEYS
            ]
        ),

    "ALL_CLINICIANS":
        np.stack(
            [
                all_clinician_targets[
                    k
                ]
                for k
                in ECG_KEYS
            ]
        ),
}

gaze_cv = (
    KFold(
        n_splits=
        N_GAZE_FOLDS,
        shuffle=True,
        random_state=
        SEED
    )
)

# Same folds for both gaze-target variants
gaze_splits = list(
    gaze_cv.split(
        np.arange(
            len(
                ECG_KEYS
            )
        )
    )
)

gaze_cv_rows = []


for target_name, Y_all in (
    TARGETS.items()
):

    print(
        "\nTARGET:",
        target_name
    )

    for fold, (
        train_idx,
        test_idx
    ) in enumerate(
        gaze_splits,
        start=1
    ):

        print(
            f"Fold {fold}: "
            f"train ECGs="
            f"{[ECG_KEYS[i] for i in train_idx]}, "
            f"held-out="
            f"{[ECG_KEYS[i] for i in test_idx]}"
        )

        fold_model, history = (
            train_clean_gaze_model(
                X_gaze[
                    train_idx
                ],
                Y_all[
                    train_idx
                ],
                seed=
                SEED
                +
                fold
            )
        )

        fine_attention = (
            get_attention_distribution(
                fold_model,
                X_gaze[
                    test_idx
                ]
            )
        )

        for local_j, global_i in enumerate(
            test_idx
        ):

            target = (
                Y_all[
                    global_i
                ]
            )

            base_att = (
                base_gaze_attention[
                    global_i
                ]
            )

            fine_att = (
                fine_attention[
                    local_j
                ]
            )

            gaze_cv_rows.append(
                {
                    "target":
                        target_name,

                    "fold":
                        fold,

                    "ECG":
                        ECG_KEYS[
                            global_i
                        ],

                    "base_cosine":
                        cosine_similarity_np(
                            base_att,
                            target
                        ),

                    "fine_cosine":
                        cosine_similarity_np(
                            fine_att,
                            target
                        ),

                    "base_MSE":
                        mse_np(
                            base_att,
                            target
                        ),

                    "fine_MSE":
                        mse_np(
                            fine_att,
                            target
                        ),

                    "base_JSD":
                        js_divergence_np(
                            base_att,
                            target
                        ),

                    "fine_JSD":
                        js_divergence_np(
                            fine_att,
                            target
                        ),

                    "final_training_loss":
                        history[
                            -1
                        ],
                }
            )

        del fold_model
        gc.collect()


gaze_cv_df = pd.DataFrame(
    gaze_cv_rows
)

gaze_cv_df.to_csv(
    os.path.join(
        OUT_DIR,
        "gaze_5fold_heldout_results.csv"
    ),
    index=False
)


# =================================================================================================
# 12. GAZE VALIDATION STATISTICS
# =================================================================================================

gaze_stat_rows = []

for target_name in (
    TARGETS.keys()
):

    sub = gaze_cv_df[
        gaze_cv_df[
            "target"
        ]
        ==
        target_name
    ]

    # Two-sided paired Wilcoxon tests
    cos_test = wilcoxon(
        sub[
            "fine_cosine"
        ],
        sub[
            "base_cosine"
        ],
        alternative=
        "two-sided"
    )

    mse_test = wilcoxon(
        sub[
            "fine_MSE"
        ],
        sub[
            "base_MSE"
        ],
        alternative=
        "two-sided"
    )

    jsd_test = wilcoxon(
        sub[
            "fine_JSD"
        ],
        sub[
            "base_JSD"
        ],
        alternative=
        "two-sided"
    )

    gaze_stat_rows.append(
        {
            "target":
                target_name,

            "base_cosine_mean":
                sub[
                    "base_cosine"
                ].mean(),

            "fine_cosine_mean":
                sub[
                    "fine_cosine"
                ].mean(),

            "cosine_delta":
                (
                    sub[
                        "fine_cosine"
                    ]
                    -
                    sub[
                        "base_cosine"
                    ]
                ).mean(),

            "cosine_Wilcoxon_p":
                cos_test.pvalue,

            "base_MSE_mean":
                sub[
                    "base_MSE"
                ].mean(),

            "fine_MSE_mean":
                sub[
                    "fine_MSE"
                ].mean(),

            "MSE_delta":
                (
                    sub[
                        "fine_MSE"
                    ]
                    -
                    sub[
                        "base_MSE"
                    ]
                ).mean(),

            "MSE_Wilcoxon_p":
                mse_test.pvalue,

            "base_JSD_mean":
                sub[
                    "base_JSD"
                ].mean(),

            "fine_JSD_mean":
                sub[
                    "fine_JSD"
                ].mean(),

            "JSD_delta":
                (
                    sub[
                        "fine_JSD"
                    ]
                    -
                    sub[
                        "base_JSD"
                    ]
                ).mean(),

            "JSD_Wilcoxon_p":
                jsd_test.pvalue,
        }
    )


gaze_stats_df = pd.DataFrame(
    gaze_stat_rows
)

print(
    "\nGAZE HELD-OUT SUMMARY"
)

print(
    gaze_stats_df.to_string(
        index=False
    )
)

gaze_stats_df.to_csv(
    os.path.join(
        OUT_DIR,
        "gaze_5fold_heldout_summary.csv"
    ),
    index=False
)


# =================================================================================================
# 13. PLOT HELD-OUT GAZE ALIGNMENT
# =================================================================================================

for target_name in (
    TARGETS.keys()
):

    sub = (
        gaze_cv_df[
            gaze_cv_df[
                "target"
            ]
            ==
            target_name
        ]
        .set_index(
            "ECG"
        )
        .loc[
            ECG_KEYS
        ]
    )

    x = np.arange(
        len(
            ECG_KEYS
        )
    )

    plt.figure(
        figsize=(
            12,
            5
        )
    )

    plt.plot(
        x,
        sub[
            "base_cosine"
        ].values,
        marker="o",
        label="Base"
    )

    plt.plot(
        x,
        sub[
            "fine_cosine"
        ].values,
        marker="o",
        label="Gaze-guided"
    )

    plt.xticks(
        x,
        ECG_KEYS,
        rotation=45,
        ha="right"
    )

    plt.ylabel(
        "Cosine similarity to held-out gaze target"
    )

    plt.title(
        f"Held-out gaze alignment: {target_name}"
    )

    plt.legend()

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUT_DIR,
            f"heldout_gaze_alignment_{target_name}.png"
        ),
        dpi=180
    )

    plt.close()


# =================================================================================================
# 14. FINAL CLEAN TRAINING ON ALL TEN GAZE ECGs
# =================================================================================================

print(
    "\n" + "=" * 100
)

print(
    "FINAL CLEAN MODELS"
)

print(
    "=" * 100
)

final_models = {}

final_histories = {}

for target_name, Y_all in (
    TARGETS.items()
):

    print(
        "\nTraining final",
        target_name
    )

    model, history = (
        train_clean_gaze_model(
            X_gaze,
            Y_all,
            seed=SEED
        )
    )

    final_models[
        target_name
    ] = model

    final_histories[
        target_name
    ] = history

    weight_path = os.path.join(
        OUT_DIR,
        (
            "clean_consultant.weights.h5"
            if target_name
            ==
            "CONSULTANT"
            else
            "clean_all_clinicians.weights.h5"
        )
    )

    model.save_weights(
        weight_path
    )

    print(
        "Saved:",
        weight_path
    )

    print(
        "Final gaze loss:",
        history[-1]
    )


# =================================================================================================
# 15. VERIFY THAT ONLY THE INTENDED LAYER CHANGED
# =================================================================================================

fresh_base = fresh_base_model()


def changed_layers_vs_base(
    model,
    base_model
):

    changed = []

    for layer in model.layers:

        try:
            base_layer = (
                base_model.get_layer(
                    layer.name
                )
            )
        except Exception:
            continue

        w1 = layer.get_weights()
        w0 = base_layer.get_weights()

        if (
            len(w1)
            !=
            len(w0)
        ):
            changed.append(
                layer.name
            )
            continue

        layer_changed = False

        for a, b in zip(
            w1,
            w0
        ):

            if (
                a.shape
                !=
                b.shape
            ):
                layer_changed = True
                break

            if (
                not np.array_equal(
                    a,
                    b
                )
            ):
                layer_changed = True
                break

        if layer_changed:
            changed.append(
                layer.name
            )

    return changed


for target_name, model in (
    final_models.items()
):

    changed = (
        changed_layers_vs_base(
            model,
            fresh_base
        )
    )

    print(
        "\nChanged layers for",
        target_name,
        ":",
        changed
    )

    if (
        set(
            changed
        )
        !=
        {
            TRAINABLE_LAYER
        }
    ):
        print(
            "WARNING: expected only",
            TRAINABLE_LAYER,
            "to change."
        )


# =================================================================================================
# 16. FINAL ATTENTION DISTRIBUTIONS ON THE TEN GAZE ECGs
# =================================================================================================

final_attention_rows = []

base_att = (
    get_attention_distribution(
        fresh_base,
        X_gaze
    )
)

consultant_att = (
    get_attention_distribution(
        final_models[
            "CONSULTANT"
        ],
        X_gaze
    )
)

all_clinician_att = (
    get_attention_distribution(
        final_models[
            "ALL_CLINICIANS"
        ],
        X_gaze
    )
)

for i, ecg in enumerate(
    ECG_KEYS
):

    for j, lead in enumerate(
        STANDARD_LEADS
    ):

        final_attention_rows.append(
            {
                "ECG":
                    ecg,

                "lead":
                    lead,

                "base_attention":
                    base_att[
                        i,
                        j
                    ],

                "consultant_attention":
                    consultant_att[
                        i,
                        j
                    ],

                "all_clinician_attention":
                    all_clinician_att[
                        i,
                        j
                    ],

                "consultant_target":
                    TARGETS[
                        "CONSULTANT"
                    ][
                        i,
                        j
                    ],

                "all_clinician_target":
                    TARGETS[
                        "ALL_CLINICIANS"
                    ][
                        i,
                        j
                    ],
            }
        )


pd.DataFrame(
    final_attention_rows
).to_csv(
    os.path.join(
        OUT_DIR,
        "final_gaze_attention_distributions.csv"
    ),
    index=False
)


# =================================================================================================
# 17. DOWNSTREAM FEATURE EXTRACTORS
# =================================================================================================

evaluation_models = {
    "BASE":
        fresh_base,

    "CONSULTANT":
        final_models[
            "CONSULTANT"
        ],

    "ALL_CLINICIANS":
        final_models[
            "ALL_CLINICIANS"
        ],
}

feature_extractors = {}

for name, model in (
    evaluation_models.items()
):

    feature_extractors[
        name
    ] = Model(
        inputs=model.input,
        outputs=model.get_layer(
            FEATURE_LAYER
        ).output
    )


def predict_all_features(
    X
):

    result = {}

    for name, extractor in (
        feature_extractors.items()
    ):

        result[
            name
        ] = (
            extractor.predict(
                X,
                batch_size=
                PRED_BATCH_SIZE,
                verbose=0
            )
        )

    return result


# =================================================================================================
# 18. GENERAL EVALUATION HELPERS
# =================================================================================================

def binary_metrics(
    y,
    pred,
    prob
):

    tn, fp, fn, tp = (
        confusion_matrix(
            y,
            pred,
            labels=[
                0,
                1
            ]
        )
        .ravel()
    )

    sensitivity = (
        tp
        /
        (
            tp + fn
        )
        if (
            tp + fn
        )
        else np.nan
    )

    specificity = (
        tn
        /
        (
            tn + fp
        )
        if (
            tn + fp
        )
        else np.nan
    )

    return {
        "accuracy":
            accuracy_score(
                y,
                pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y,
                pred
            ),

        "sensitivity":
            sensitivity,

        "specificity":
            specificity,

        "F1":
            f1_score(
                y,
                pred,
                zero_division=0
            ),

        "AUROC":
            roc_auc_score(
                y,
                prob
            ),

        "PRAUC":
            average_precision_score(
                y,
                prob
            ),

        "TP":
            int(tp),

        "TN":
            int(tn),

        "FP":
            int(fp),

        "FN":
            int(fn),
    }


def paired_bootstrap_differences(
    y,
    base_pred,
    base_prob,
    model_pred,
    model_prob,
    clusters=None,
    n_boot=BOOT_N,
    seed=SEED
):

    rng = (
        np.random.default_rng(
            seed
        )
    )

    acc = []
    bal = []
    auc = []
    prauc = []

    n = len(
        y
    )

    if clusters is not None:

        clusters = np.asarray(
            clusters
        )

        unique_clusters = (
            np.unique(
                clusters
            )
        )

    for _ in range(
        n_boot
    ):

        if clusters is None:

            idx = rng.integers(
                0,
                n,
                size=n
            )

        else:

            sampled_clusters = (
                rng.choice(
                    unique_clusters,
                    size=len(
                        unique_clusters
                    ),
                    replace=True
                )
            )

            pieces = [
                np.where(
                    clusters == c
                )[0]
                for c
                in sampled_clusters
            ]

            idx = np.concatenate(
                pieces
            )

        yy = y[
            idx
        ]

        if (
            len(
                np.unique(
                    yy
                )
            )
            <
            2
        ):
            continue

        bp = base_pred[
            idx
        ]

        bprob = base_prob[
            idx
        ]

        mp = model_pred[
            idx
        ]

        mprob = model_prob[
            idx
        ]

        acc.append(
            accuracy_score(
                yy,
                mp
            )
            -
            accuracy_score(
                yy,
                bp
            )
        )

        bal.append(
            balanced_accuracy_score(
                yy,
                mp
            )
            -
            balanced_accuracy_score(
                yy,
                bp
            )
        )

        auc.append(
            roc_auc_score(
                yy,
                mprob
            )
            -
            roc_auc_score(
                yy,
                bprob
            )
        )

        prauc.append(
            average_precision_score(
                yy,
                mprob
            )
            -
            average_precision_score(
                yy,
                bprob
            )
        )

    def ci(
        values
    ):

        return (
            float(
                np.percentile(
                    values,
                    2.5
                )
            ),
            float(
                np.percentile(
                    values,
                    97.5
                )
            )
        )

    return {
        "accuracy_diff":
            accuracy_score(
                y,
                model_pred
            )
            -
            accuracy_score(
                y,
                base_pred
            ),

        "accuracy_CI":
            ci(
                acc
            ),

        "balanced_accuracy_diff":
            balanced_accuracy_score(
                y,
                model_pred
            )
            -
            balanced_accuracy_score(
                y,
                base_pred
            ),

        "balanced_accuracy_CI":
            ci(
                bal
            ),

        "AUROC_diff":
            roc_auc_score(
                y,
                model_prob
            )
            -
            roc_auc_score(
                y,
                base_prob
            ),

        "AUROC_CI":
            ci(
                auc
            ),

        "PRAUC_diff":
            average_precision_score(
                y,
                model_prob
            )
            -
            average_precision_score(
                y,
                base_prob
            ),

        "PRAUC_CI":
            ci(
                prauc
            ),
    }


def exact_mcnemar(
    y,
    base_pred,
    model_pred
):

    base_correct = (
        base_pred
        ==
        y
    )

    model_correct = (
        model_pred
        ==
        y
    )

    b = int(
        np.sum(
            base_correct
            &
            ~model_correct
        )
    )

    c = int(
        np.sum(
            ~base_correct
            &
            model_correct
        )
    )

    if (
        b + c
        ==
        0
    ):
        return (
            b,
            c,
            1.0
        )

    p = binomtest(
        min(
            b,
            c
        ),
        n=(
            b + c
        ),
        p=0.5,
        alternative=
        "two-sided"
    ).pvalue

    return (
        b,
        c,
        float(
            p
        )
    )


def holm_adjust(
    pvalues
):

    pvalues = (
        np.asarray(
            pvalues,
            dtype=float
        )
    )

    m = len(
        pvalues
    )

    order = np.argsort(
        pvalues
    )

    adjusted_sorted = (
        np.zeros(
            m
        )
    )

    running_max = 0.0

    for rank, idx in enumerate(
        order
    ):

        multiplier = (
            m - rank
        )

        value = min(
            1.0,
            pvalues[
                idx
            ]
            *
            multiplier
        )

        running_max = max(
            running_max,
            value
        )

        adjusted_sorted[
            rank
        ] = running_max

    adjusted = (
        np.zeros(
            m
        )
    )

    for rank, idx in enumerate(
        order
    ):

        adjusted[
            idx
        ] = (
            adjusted_sorted[
                rank
            ]
        )

    return adjusted


def run_oof_evaluation(
    task_name,
    y,
    feature_dict,
    record_ids,
    groups=None,
    grouped=False,
    family_name="evaluation"
):

    y = np.asarray(
        y,
        dtype=int
    )

    record_ids = np.asarray(
        record_ids
    )

    if groups is not None:
        groups = np.asarray(
            groups
        )

    print(
        "\n" + "=" * 100
    )

    print(
        family_name,
        ":",
        task_name
    )

    print(
        "N =",
        len(
            y
        ),
        "positive =",
        int(
            y.sum()
        ),
        "negative =",
        int(
            len(y)
            -
            y.sum()
        )
    )

    if (
        grouped
        and
        groups is not None
        and
        HAVE_STRATIFIED_GROUP_KFOLD
    ):

        splitter = (
            StratifiedGroupKFold(
                n_splits=
                N_EVAL_FOLDS,
                shuffle=True,
                random_state=
                SEED
            )
        )

        split_iterator = (
            splitter.split(
                np.zeros(
                    len(
                        y
                    )
                ),
                y,
                groups=
                groups
            )
        )

    else:

        if (
            grouped
            and
            groups is not None
            and
            not HAVE_STRATIFIED_GROUP_KFOLD
        ):
            print(
                "WARNING: StratifiedGroupKFold unavailable; "
                "falling back to StratifiedKFold."
            )

        splitter = (
            StratifiedKFold(
                n_splits=
                N_EVAL_FOLDS,
                shuffle=True,
                random_state=
                SEED
            )
        )

        split_iterator = (
            splitter.split(
                np.zeros(
                    len(
                        y
                    )
                ),
                y
            )
        )

    splits = list(
        split_iterator
    )

    oof = {
        name: {
            "pred":
                np.zeros(
                    len(
                        y
                    ),
                    dtype=int
                ),

            "prob":
                np.zeros(
                    len(
                        y
                    ),
                    dtype=float
                ),
        }
        for name
        in feature_dict
    }

    fold_id = np.zeros(
        len(
            y
        ),
        dtype=int
    )

    for fold, (
        tr,
        te
    ) in enumerate(
        splits,
        start=1
    ):

        fold_id[
            te
        ] = fold

        print(
            f"Fold {fold}: "
            f"train={len(tr)}, "
            f"test={len(te)}, "
            f"test positives={int(y[te].sum())}"
        )

        for model_name, Z in (
            feature_dict.items()
        ):

            clf = (
                LogisticRegression(
                    max_iter=3000,
                    solver="lbfgs",
                    class_weight=
                    "balanced",
                    random_state=
                    SEED
                )
            )

            clf.fit(
                Z[
                    tr
                ],
                y[
                    tr
                ]
            )

            oof[
                model_name
            ][
                "pred"
            ][
                te
            ] = (
                clf.predict(
                    Z[
                        te
                    ]
                )
            )

            oof[
                model_name
            ][
                "prob"
            ][
                te
            ] = (
                clf.predict_proba(
                    Z[
                        te
                    ]
                )[
                    :,
                    1
                ]
            )

    summary_rows = []

    prediction_rows = []

    for model_name in [
        "BASE",
        "CONSULTANT",
        "ALL_CLINICIANS",
    ]:

        pred = (
            oof[
                model_name
            ][
                "pred"
            ]
        )

        prob = (
            oof[
                model_name
            ][
                "prob"
            ]
        )

        metrics = (
            binary_metrics(
                y,
                pred,
                prob
            )
        )

        print(
            "\n",
            model_name,
            metrics
        )

        row = {
            "family":
                family_name,

            "task":
                task_name,

            "model":
                model_name,

            "N":
                len(
                    y
                ),

            "positive":
                int(
                    y.sum()
                ),

            "negative":
                int(
                    len(y)
                    -
                    y.sum()
                ),
        }

        row.update(
            metrics
        )

        summary_rows.append(
            row
        )

        for i in range(
            len(
                y
            )
        ):

            prediction_rows.append(
                {
                    "family":
                        family_name,

                    "task":
                        task_name,

                    "model":
                        model_name,

                    "record_id":
                        record_ids[
                            i
                        ],

                    "fold":
                        int(
                            fold_id[
                                i
                            ]
                        ),

                    "true_label":
                        int(
                            y[
                                i
                            ]
                        ),

                    "predicted_label":
                        int(
                            pred[
                                i
                            ]
                        ),

                    "probability":
                        float(
                            prob[
                                i
                            ]
                        ),
                }
            )

    paired_rows = []

    base_pred = (
        oof[
            "BASE"
        ][
            "pred"
        ]
    )

    base_prob = (
        oof[
            "BASE"
        ][
            "prob"
        ]
    )

    for model_name in [
        "CONSULTANT",
        "ALL_CLINICIANS",
    ]:

        pred = (
            oof[
                model_name
            ][
                "pred"
            ]
        )

        prob = (
            oof[
                model_name
            ][
                "prob"
            ]
        )

        diff = (
            paired_bootstrap_differences(
                y,
                base_pred,
                base_prob,
                pred,
                prob,
                clusters=(
                    groups
                    if grouped
                    else None
                ),
                n_boot=
                BOOT_N,
                seed=
                SEED
            )
        )

        b, c, p = (
            exact_mcnemar(
                y,
                base_pred,
                pred
            )
        )

        paired_rows.append(
            {
                "family":
                    family_name,

                "task":
                    task_name,

                "comparison":
                    (
                        model_name
                        +
                        "_vs_BASE"
                    ),

                "accuracy_difference":
                    diff[
                        "accuracy_diff"
                    ],

                "accuracy_CI_low":
                    diff[
                        "accuracy_CI"
                    ][
                        0
                    ],

                "accuracy_CI_high":
                    diff[
                        "accuracy_CI"
                    ][
                        1
                    ],

                "balanced_accuracy_difference":
                    diff[
                        "balanced_accuracy_diff"
                    ],

                "balanced_accuracy_CI_low":
                    diff[
                        "balanced_accuracy_CI"
                    ][
                        0
                    ],

                "balanced_accuracy_CI_high":
                    diff[
                        "balanced_accuracy_CI"
                    ][
                        1
                    ],

                "AUROC_difference":
                    diff[
                        "AUROC_diff"
                    ],

                "AUROC_CI_low":
                    diff[
                        "AUROC_CI"
                    ][
                        0
                    ],

                "AUROC_CI_high":
                    diff[
                        "AUROC_CI"
                    ][
                        1
                    ],

                "PRAUC_difference":
                    diff[
                        "PRAUC_diff"
                    ],

                "PRAUC_CI_low":
                    diff[
                        "PRAUC_CI"
                    ][
                        0
                    ],

                "PRAUC_CI_high":
                    diff[
                        "PRAUC_CI"
                    ][
                        1
                    ],

                "McNemar_base_only_correct":
                    b,

                "McNemar_model_only_correct":
                    c,

                "McNemar_p":
                    p,
            }
        )

    return (
        pd.DataFrame(
            summary_rows
        ),
        pd.DataFrame(
            paired_rows
        ),
        pd.DataFrame(
            prediction_rows
        )
    )


# =================================================================================================
# 19. PTB-XL INTERNAL REPRESENTATION EVALUATION
# =================================================================================================

print(
    "\n" + "=" * 100
)

print(
    "PTB-XL INTERNAL EVALUATION"
)

print(
    "=" * 100
)

ptb_meta = pd.read_csv(
    os.path.join(
        PTB_ROOT,
        "ptbxl_database.csv"
    )
)

scp = pd.read_csv(
    os.path.join(
        PTB_ROOT,
        "scp_statements.csv"
    ),
    index_col=0
)


def parse_scp_codes(
    value
):

    try:
        return set(
            ast.literal_eval(
                value
            ).keys()
        )
    except Exception:
        return set()


ptb_codes = (
    ptb_meta[
        "scp_codes"
    ]
    .apply(
        parse_scp_codes
    )
)


# MI
mi_codes = set(
    scp.loc[
        scp[
            "diagnostic_class"
        ]
        ==
        "MI"
    ].index
)

ptb_is_mi = (
    ptb_codes.apply(
        lambda x:
        bool(
            x
            &
            mi_codes
        )
    )
)

ptb_is_norm = (
    ptb_codes.apply(
        lambda x:
        "NORM"
        in
        x
    )
)

mi_mask = (
    ptb_is_mi
    ^
    ptb_is_norm
)


# AF vs sinus rhythm
ptb_is_af = (
    ptb_codes.apply(
        lambda x:
        "AFIB"
        in
        x
    )
)

ptb_is_sr = (
    ptb_codes.apply(
        lambda x:
        "SR"
        in
        x
    )
)

af_mask = (
    ptb_is_af
    ^
    ptb_is_sr
)


ptb_task_defs = {
    "MI_vs_NORM": {
        "mask":
            mi_mask.values,

        "labels":
            ptb_is_mi[
                mi_mask
            ].astype(
                int
            ).values,
    },

    "AFIB_vs_SR": {
        "mask":
            af_mask.values,

        "labels":
            ptb_is_af[
                af_mask
            ].astype(
                int
            ).values,
    },
}


# Union of all records needed
needed_ptb_indices = sorted(
    set(
        np.where(
            mi_mask.values
        )[0]
    )
    |
    set(
        np.where(
            af_mask.values
        )[0]
    )
)

print(
    "Unique PTB-XL records needed:",
    len(
        needed_ptb_indices
    )
)


def load_ptb_signal_by_row(
    row
):

    path = os.path.join(
        PTB_ROOT,
        row[
            "filename_lr"
        ]
    )

    sig, fields = (
        wfdb.rdsamp(
            path
        )
    )

    names = (
        fields.get(
            "sig_name",
            []
        )
    )

    name_map = {
        str(
            name
        ).upper():
            i
        for i, name
        in enumerate(
            names
        )
    }

    if all(
        lead.upper()
        in
        name_map
        for lead
        in STANDARD_LEADS
    ):

        sig = sig[
            :,
            [
                name_map[
                    lead.upper()
                ]
                for lead
                in STANDARD_LEADS
            ]
        ]

    if (
        sig.shape[
            1
        ]
        !=
        12
    ):
        raise ValueError(
            f"Unexpected PTB shape {sig.shape}"
        )

    if (
        sig.shape[
            0
        ]
        !=
        1000
    ):

        old_x = np.linspace(
            0,
            1,
            sig.shape[
                0
            ]
        )

        new_x = np.linspace(
            0,
            1,
            1000
        )

        resampled = np.zeros(
            (
                1000,
                12
            ),
            dtype=np.float32
        )

        for j in range(
            12
        ):

            resampled[
                :,
                j
            ] = np.interp(
                new_x,
                old_x,
                sig[
                    :,
                    j
                ]
            )

        sig = resampled

    return (
        sig.T[
            ...,
            None
        ]
        .astype(
            np.float32
        )
    )


# Feature extraction in disk batches
ptb_feature_store = {
    name:
        np.zeros(
            (
                len(
                    needed_ptb_indices
                ),
                64
            ),
            dtype=np.float32
        )
    for name
    in feature_extractors
}

ptb_index_to_pos = {
    original_idx:
        position
    for position, original_idx
    in enumerate(
        needed_ptb_indices
    )
}


print(
    "Extracting PTB-XL features..."
)

for start in range(
    0,
    len(
        needed_ptb_indices
    ),
    PRED_BATCH_SIZE
):

    batch_indices = (
        needed_ptb_indices[
            start:
            start
            +
            PRED_BATCH_SIZE
        ]
    )

    X_batch = np.stack(
        [
            load_ptb_signal_by_row(
                ptb_meta.iloc[
                    idx
                ]
            )
            for idx
            in batch_indices
        ]
    )

    for name, extractor in (
        feature_extractors.items()
    ):

        Z = extractor.predict(
            X_batch,
            batch_size=
            PRED_BATCH_SIZE,
            verbose=0
        )

        ptb_feature_store[
            name
        ][
            start:
            start
            +
            len(
                batch_indices
            )
        ] = Z

    if (
        start
        %
        (
            PRED_BATCH_SIZE
            *
            20
        )
        ==
        0
    ):
        print(
            start,
            "/",
            len(
                needed_ptb_indices
            )
        )

    del X_batch
    gc.collect()


ptb_summary_frames = []
ptb_paired_frames = []
ptb_prediction_frames = []


for task_name, task_info in (
    ptb_task_defs.items()
):

    mask = task_info[
        "mask"
    ]

    original_indices = (
        np.where(
            mask
        )[0]
    )

    y = task_info[
        "labels"
    ]

    positions = np.array(
        [
            ptb_index_to_pos[
                idx
            ]
            for idx
            in original_indices
        ]
    )

    task_features = {
        name:
            Z[
                positions
            ]
        for name, Z
        in ptb_feature_store.items()
    }

    task_rows = (
        ptb_meta.iloc[
            original_indices
        ]
        .reset_index(
            drop=True
        )
    )

    record_ids = (
        task_rows[
            "ecg_id"
        ].values
        if
        "ecg_id"
        in
        task_rows.columns
        else
        original_indices
    )

    patient_groups = (
        task_rows[
            "patient_id"
        ].values
        if
        "patient_id"
        in
        task_rows.columns
        else
        None
    )

    summary_df, paired_df, pred_df = (
        run_oof_evaluation(
            task_name=
            task_name,

            y=
            y,

            feature_dict=
            task_features,

            record_ids=
            record_ids,

            groups=
            patient_groups,

            grouped=(
                patient_groups
                is not None
            ),

            family_name=
            "PTBXL_internal"
        )
    )

    ptb_summary_frames.append(
        summary_df
    )

    ptb_paired_frames.append(
        paired_df
    )

    ptb_prediction_frames.append(
        pred_df
    )


ptb_summary = pd.concat(
    ptb_summary_frames,
    ignore_index=True
)

ptb_paired = pd.concat(
    ptb_paired_frames,
    ignore_index=True
)

ptb_predictions = pd.concat(
    ptb_prediction_frames,
    ignore_index=True
)

ptb_paired[
    "McNemar_p_Holm"
] = holm_adjust(
    ptb_paired[
        "McNemar_p"
    ].values
)


ptb_summary.to_csv(
    os.path.join(
        OUT_DIR,
        "PTBXL_clean_OOF_summary.csv"
    ),
    index=False
)

ptb_paired.to_csv(
    os.path.join(
        OUT_DIR,
        "PTBXL_clean_OOF_paired.csv"
    ),
    index=False
)

ptb_predictions.to_csv(
    os.path.join(
        OUT_DIR,
        "PTBXL_clean_OOF_predictions.csv"
    ),
    index=False
)


# =================================================================================================
# 20. LUDB EXTERNAL VALIDATION
# =================================================================================================

print(
    "\n" + "=" * 100
)

print(
    "LUDB EXTERNAL VALIDATION"
)

print(
    "=" * 100
)

ludb_meta = pd.read_csv(
    LUDB_CSV
)


def clean_text(
    series
):

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


def nonempty(
    series
):

    x = clean_text(
        series
    )

    return (
        x.ne("")
        &
        ~x.str.fullmatch(
            "nan",
            case=False
        )
    )


rhythm = clean_text(
    ludb_meta[
        "Rhythms"
    ]
)

ischemia_field = clean_text(
    ludb_meta[
        "Ischemia"
    ]
)

conduction_field = clean_text(
    ludb_meta[
        "Conduction abnormalities"
    ]
)

hypertrophy_field = clean_text(
    ludb_meta[
        "Hypertrophies"
    ]
)

pacing_field = clean_text(
    ludb_meta[
        "Cardiac pacing"
    ]
)


# -------------------------------------------------------------------------------------------------
# CLEAN EXTERNAL PHENOTYPE DEFINITIONS
# -------------------------------------------------------------------------------------------------

# AF: explicit AF annotation vs all others
af_positive = (
    rhythm.str.contains(
        "Atrial fibrillation",
        case=False,
        regex=False,
        na=False
    )
)


# Ischaemia-related:
#
# Positive:
#   active ischaemia / STEMI / NSTEMI-related terms
#
# Negative:
#   no annotation in the Ischemia field
#
# Pure scar-only annotations are excluded rather than misclassified
# as active ischaemia or normal.
#
ischemia_present = (
    nonempty(
        ludb_meta[
            "Ischemia"
        ]
    )
)

ischemia_positive = (
    ischemia_field.str.contains(
        r"ischemia|stemi|nstemi",
        case=False,
        regex=True,
        na=False
    )
)

ischemia_include = (
    (~ischemia_present)
    |
    ischemia_positive
)


# Any conduction abnormality vs absent
conduction_positive = (
    nonempty(
        ludb_meta[
            "Conduction abnormalities"
        ]
    )
)


# Explicit left ventricular hypertrophy vs all others
lvh_positive = (
    hypertrophy_field.str.contains(
        r"left ventricular hypertrophy|\bLVH\b",
        case=False,
        regex=True,
        na=False
    )
)


# Any cardiac pacing annotation vs absent
paced_positive = (
    nonempty(
        ludb_meta[
            "Cardiac pacing"
        ]
    )
)


ludb_task_defs = {
    "AFib": {
        "mask":
            np.ones(
                len(
                    ludb_meta
                ),
                dtype=bool
            ),

        "labels":
            af_positive
            .astype(
                int
            )
            .values,
    },

    "Ischaemia_related": {
        "mask":
            ischemia_include
            .values,

        "labels":
            ischemia_positive[
                ischemia_include
            ]
            .astype(
                int
            )
            .values,
    },

    "Conduction": {
        "mask":
            np.ones(
                len(
                    ludb_meta
                ),
                dtype=bool
            ),

        "labels":
            conduction_positive
            .astype(
                int
            )
            .values,
    },

    "LVH": {
        "mask":
            np.ones(
                len(
                    ludb_meta
                ),
                dtype=bool
            ),

        "labels":
            lvh_positive
            .astype(
                int
            )
            .values,
    },

    "Paced": {
        "mask":
            np.ones(
                len(
                    ludb_meta
                ),
                dtype=bool
            ),

        "labels":
            paced_positive
            .astype(
                int
            )
            .values,
    },
}


print(
    "\nLUDB CLEAN TASK COUNTS"
)

for task_name, info in (
    ludb_task_defs.items()
):

    y = info[
        "labels"
    ]

    print(
        task_name,
        "N =",
        len(
            y
        ),
        "positive =",
        int(
            y.sum()
        ),
        "negative =",
        int(
            len(y)
            -
            y.sum()
        )
    )


def load_ludb_signal(
    record_id
):

    path = os.path.join(
        LUDB_DATA,
        str(
            int(
                record_id
            )
        )
    )

    sig, fields = (
        wfdb.rdsamp(
            path
        )
    )

    names = (
        fields.get(
            "sig_name",
            []
        )
    )

    name_map = {
        str(
            name
        ).upper():
            i
        for i, name
        in enumerate(
            names
        )
    }

    if all(
        lead.upper()
        in
        name_map
        for lead
        in STANDARD_LEADS
    ):

        sig = sig[
            :,
            [
                name_map[
                    lead.upper()
                ]
                for lead
                in STANDARD_LEADS
            ]
        ]

    if (
        sig.shape[
            1
        ]
        !=
        12
    ):

        raise ValueError(
            f"LUDB record {record_id}: {sig.shape}"
        )

    old_x = np.linspace(
        0,
        1,
        sig.shape[
            0
        ]
    )

    new_x = np.linspace(
        0,
        1,
        1000
    )

    resampled = np.zeros(
        (
            1000,
            12
        ),
        dtype=np.float32
    )

    for j in range(
        12
    ):

        resampled[
            :,
            j
        ] = np.interp(
            new_x,
            old_x,
            sig[
                :,
                j
            ]
        )

    resampled = (
        resampled.T
    )

    # External waveform standardisation
    resampled = (
        resampled
        -
        resampled.mean(
            axis=1,
            keepdims=True
        )
    ) / (
        resampled.std(
            axis=1,
            keepdims=True
        )
        +
        1e-8
    )

    return (
        resampled[
            ...,
            None
        ]
        .astype(
            np.float32
        )
    )


print(
    "\nLoading LUDB..."
)

X_ludb = np.stack(
    [
        load_ludb_signal(
            record_id
        )
        for record_id
        in ludb_meta[
            "ID"
        ]
    ]
)

print(
    "LUDB X:",
    X_ludb.shape
)


ludb_features = (
    predict_all_features(
        X_ludb
    )
)


ludb_summary_frames = []
ludb_paired_frames = []
ludb_prediction_frames = []


for task_name, task_info in (
    ludb_task_defs.items()
):

    mask = np.asarray(
        task_info[
            "mask"
        ],
        dtype=bool
    )

    y = np.asarray(
        task_info[
            "labels"
        ],
        dtype=int
    )

    task_features = {
        name:
            Z[
                mask
            ]
        for name, Z
        in ludb_features.items()
    }

    record_ids = (
        ludb_meta.loc[
            mask,
            "ID"
        ]
        .values
    )

    summary_df, paired_df, pred_df = (
        run_oof_evaluation(
            task_name=
            task_name,

            y=
            y,

            feature_dict=
            task_features,

            record_ids=
            record_ids,

            groups=None,

            grouped=False,

            family_name=
            "LUDB_external"
        )
    )

    ludb_summary_frames.append(
        summary_df
    )

    ludb_paired_frames.append(
        paired_df
    )

    ludb_prediction_frames.append(
        pred_df
    )


ludb_summary = pd.concat(
    ludb_summary_frames,
    ignore_index=True
)

ludb_paired = pd.concat(
    ludb_paired_frames,
    ignore_index=True
)

ludb_predictions = pd.concat(
    ludb_prediction_frames,
    ignore_index=True
)

ludb_paired[
    "McNemar_p_Holm"
] = holm_adjust(
    ludb_paired[
        "McNemar_p"
    ].values
)


ludb_summary.to_csv(
    os.path.join(
        OUT_DIR,
        "LUDB_clean_5fold_OOF_summary.csv"
    ),
    index=False
)

ludb_paired.to_csv(
    os.path.join(
        OUT_DIR,
        "LUDB_clean_5fold_OOF_paired.csv"
    ),
    index=False
)

ludb_predictions.to_csv(
    os.path.join(
        OUT_DIR,
        "LUDB_clean_5fold_OOF_predictions.csv"
    ),
    index=False
)


# =================================================================================================
# 21. SAVE TRAINING HISTORIES
# =================================================================================================

history_rows = []

for target_name, history in (
    final_histories.items()
):

    for epoch, loss in enumerate(
        history,
        start=1
    ):

        history_rows.append(
            {
                "target":
                    target_name,

                "epoch":
                    epoch,

                "training_gaze_MSE":
                    loss,
            }
        )

pd.DataFrame(
    history_rows
).to_csv(
    os.path.join(
        OUT_DIR,
        "final_training_histories.csv"
    ),
    index=False
)


# =================================================================================================
# 22. SAVE COMPLETE EXPERIMENT CONFIGURATION
# =================================================================================================

config = {
    "seed":
        SEED,

    "base_model":
        BASE_MODEL_PATH,

    "standard_lead_order":
        STANDARD_LEADS,

    "display_lead_order":
        DISPLAY_LEADS,

    "attention_layer":
        ATTENTION_LAYER,

    "feature_layer":
        FEATURE_LAYER,

    "only_trainable_layer":
        TRAINABLE_LAYER,

    "gaze_loss":
        "mean_squared_error",

    "classification_loss":
        None,

    "learning_rate":
        LEARNING_RATE,

    "epochs":
        EPOCHS,

    "augmentations_per_ECG":
        AUGMENTATIONS_PER_ECG,

    "noise_SD":
        NOISE_SD,

    "training_batch_size":
        TRAIN_BATCH_SIZE,

    "gaze_cross_validation":
        "5-fold ECG-level held-out",

    "consultant_readers":
        9,

    "all_clinician_readers":
        20,

    "all_clinician_groups":
        [
            "Consultant",
            "Fellow",
            "resident",
        ],

    "reader_aggregation":
        (
            "Each reader's 12-lead dwell-time vector "
            "normalised separately, then readers averaged equally."
        ),

    "PTBXL_evaluation":
        (
            "5-fold OOF logistic-regression probing; "
            "patient-grouped where patient_id available"
        ),

    "LUDB_evaluation":
        (
            "5-fold OOF logistic-regression probing "
            "using dedicated metadata columns"
        ),

    "bootstrap_repetitions":
        BOOT_N,

    "grid_bounds_fraction":
        GRID_BOUNDS_FRAC,

    "cell_pad_x":
        CELL_PAD_X,

    "cell_pad_y":
        CELL_PAD_Y,

    "pseudo_signal_method":
        (
            "Per-lead 3x4 image crop; row/column background "
            "suppression; vertical centre-of-mass of residual "
            "dark trace; interpolation; Savitzky-Golay smoothing; "
            "per-lead z-normalisation."
        ),
}

with open(
    os.path.join(
        OUT_DIR,
        "experiment_config.json"
    ),
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )


# =================================================================================================
# 23. FINAL MANUSCRIPT-READY OUTPUT
# =================================================================================================

print(
    "\n\n" + "=" * 110
)

print(
    "FINAL CLEAN GAZE-ALIGNMENT VALIDATION"
)

print(
    "=" * 110
)

print(
    "\nHELD-OUT GAZE ALIGNMENT"
)

print(
    gaze_stats_df.to_string(
        index=False
    )
)


print(
    "\n\nPTB-XL INTERNAL OOF RESULTS"
)

print(
    ptb_summary.to_string(
        index=False
    )
)


print(
    "\n\nPTB-XL PAIRED COMPARISONS"
)

print(
    ptb_paired.to_string(
        index=False
    )
)


print(
    "\n\nLUDB EXTERNAL OOF RESULTS"
)

print(
    ludb_summary.to_string(
        index=False
    )
)


print(
    "\n\nLUDB PAIRED COMPARISONS"
)

print(
    ludb_paired.to_string(
        index=False
    )
)


print(
    "\n\n" + "=" * 110
)

print(
    "EXPERIMENT COMPLETE"
)

print(
    "=" * 110
)

print(
    "\nOutputs saved to:",
    OUT_DIR
)

print(
    """
Key files:

  gaze_5fold_heldout_results.csv
  gaze_5fold_heldout_summary.csv

  clean_consultant.weights.h5
  clean_all_clinicians.weights.h5

  final_gaze_attention_distributions.csv

  PTBXL_clean_OOF_summary.csv
  PTBXL_clean_OOF_paired.csv
  PTBXL_clean_OOF_predictions.csv

  LUDB_clean_5fold_OOF_summary.csv
  LUDB_clean_5fold_OOF_paired.csv
  LUDB_clean_5fold_OOF_predictions.csv

  final_training_histories.csv
  experiment_config.json

  QC_*.png
"""
)